In [0]:
from pyspark.sql import functions as F

PATH = "/Volumes/vidaplus/ingestion/raw/fact" 

TABLE_FACT_ATENDIMENTOS = "vidaplus.bronze.fact_atendimentos"

In [0]:
def read_raw_file(file: str):
    return(
        spark.read.format("csv")
        .option("sep", ",")
        .option("header", True)
        .option("quote", '"')
        .option("escape", '"')
        .option("mode", "PERMISSIVE")
        .option("encoding", "UTF-8")
        .load(f"{PATH}/{file}")
        .withColumns({
            "_arquivo_origem": F.col("_metadata.file_name"),
            "_ingerido_em": F.current_timestamp(),
        })
    )    

def load_bronze(file: str, table: str):
    return (
        read_raw_file(file).write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(table)
    )    


In [0]:
load_bronze("f1_atendimentos*.csv", TABLE_FACT_ATENDIMENTOS)

count = spark.table(TABLE_FACT_ATENDIMENTOS).count()
print(f"SUCCESS | {TABLE_FACT_ATENDIMENTOS}: {count:,} registers")

In [0]:
table_comments = [
    (TABLE_FACT_ATENDIMENTOS,
     "Bronze - registros de atendimentos hospitalares e ambulatoriais, como chegou do CSV. "
     "Contem dados de patient flow (entrada, classificacao de risco, internacao, UTI, ventilacao mecanica), "
     "desfecho clinico, custo estimado e CID-10. Fonte: PEP/HIS/RIS/LIS. "
     "Chave: id_atendimento."),
]

for table, comment in table_comments:
    spark.sql(f"COMMENT ON TABLE {table} IS '{comment}'")

print(f"Table comments applied: {len(table_comments)} tables")

In [0]:
validation_failures = []

validation_config = {
    TABLE_FACT_ATENDIMENTOS: {"expected_cols": 30, "key_cols": ["id_atendimento", "data_admissao", "id_unidade"], "csv_file": "f1_atendimentos*.csv"},
}

for table_path, cfg in validation_config.items():
    try:
        df = spark.table(table_path)
        row_count = df.count()
        actual_cols = len(df.columns)
        checks = []

        if row_count == 0:
            checks.append("Empty Table")
        else:
            if actual_cols != cfg["expected_cols"]:
                checks.append(f"{actual_cols} columns (expected {cfg['expected_cols']})")
            for col in cfg["key_cols"]:
                if df.filter(F.col(col).isNull()).count() == row_count:
                    checks.append(f"column {col} 100% null, file probably corrupted")
            csv_count = read_raw_file(cfg["csv_file"]).count()
            if csv_count != row_count:
                checks.append(f"row count mismatch (CSV={csv_count:,} vs Delta={row_count:,})")

        if checks:
            validation_failures.append(f"{table_path}: {'; '.join(checks)}")
        else:
            print(f"VALID | {table_path}: {row_count:,} register(s), {actual_cols} columns, PK ok, CSV reconciled")
    except Exception as e:
        validation_failures.append(f"{table_path}: {str(e)}")

if validation_failures:
    raise RuntimeError("Failures:\n" + "\n".join(validation_failures))
